<a href="https://colab.research.google.com/github/jonahduffy/soccer-scouting-player-predictive-machine-learning-model/blob/main/MLVpythoncode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Check available columns
print(list(df.columns))

['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'xG', 'npxG', 'xAG', 'npxG+xAG', 'PrgC', 'PrgP', 'PrgR', 'G+A-PK', 'xG+xAG', 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting', '90s_stats_shooting', 'Gls_stats_shooting', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'Dist', 'FK', 'PK_stats_shooting', 'PKatt_stats_shooting', 'xG_stats_shooting', 'npxG_stats_shooting', 'npxG/Sh', 'G-xG', 'np:G-xG', 'Rk_stats_passing', 'Nation_stats_passing', 'Pos_stats_passing', 'Comp_stats_passing', 'Age_stats_passing', 'Born_stats_passing', '90s_stats_passing', 'Cmp', 'Att', 'Cmp%', 'TotDist', 'PrgDist', 'Ast_stats_passing', 'xAG_stats_passing', 'xA', 'A-xAG', 'KP', '1/3', 'PPA', 'CrsPA', 'PrgP_stats_passing', 'Rk_stats_passing_types', 'Nation_stats_passing_types', 'Pos_stats_passing_types', 'Comp_stats

In [2]:
# --- CELL 1: SETUP ---
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Load the database
# MAKE SURE the file name matches exactly what you uploaded on the left sidebar!
filename = 'players_data-2024_2025.csv'

try:
    df = pd.read_csv(filename)
    print("✅ Success! Database loaded.")
    print(f"   Found {len(df)} players.")
except FileNotFoundError:
    print("❌ Error: File not found.")
    print("   1. Click the Folder icon 📁 on the left.")
    print("   2. Drag and drop your .csv file there.")
    print("   3. Make sure the filename matches the code exactly.")

✅ Success! Database loaded.
   Found 2854 players.


In [4]:
# --- CELL 2: TRAIN THE AI (UPDATED) ---

# EXPANDED STATS LIST
# We are now using 14 distinct metrics for a full "360 Degree" view of the player
features = [
    'Gls', 'Ast',       # Scoring
    'xG', 'npxG',       # Expected Goals (Quality of chances)
    'PrgC', 'PrgP',     # Progression (Carries & Passes)
    'Tkl', 'Int',       # Defense (Tackles & Interceptions)
    'Blocks', 'Clr',    # Defense (Blocks & Clearances)
    'AerWon',           # Physical (Aerial Duels Won)
    'SCA',              # Shot Creating Actions (Creativity)
    'Cmp%',             # Passing Accuracy
    'DrbSucc%',         # Dribbling Success %
    'Touches',          # Involvement in the game
    'Min'               # Playing Time
]

# Clean the data
# We check if these columns exist first to avoid crashing
available_features = [col for col in features if col in df.columns]

# Filter data
df_clean = df[df['Min'] > 500].copy()
df_clean[available_features] = df_clean[available_features].fillna(0)

# Normalize
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_clean[available_features])

# Train Model
model = NearestNeighbors(n_neighbors=6, metric='euclidean')
model.fit(scaled_features)

print(f"✅ AI Model upgraded! Now tracking {len(available_features)} different stats.")

✅ AI Model upgraded! Now tracking 14 different stats.


In [5]:
# --- CELL 3: SCOUTING TOOL (CLEAN LIST VERSION) ---
def find_bargain_players(player_name):
    # 1. Find the target player in the database
    target = df_clean[df_clean['Player'].str.contains(player_name, case=False, na=False)]

    if target.empty:
        print(f"❌ Player '{player_name}' not found.")
        return

    # Get the target's stats for calculation
    target_idx = target.index[0]
    target_stats = scaled_features[df_clean.index.get_loc(target_idx)].reshape(1, -1)

    # Get the target's info for display
    t_player = target.iloc[0]

    print(f"\n🔎 ANALYZING TARGET: {t_player['Player']}")
    print(f"   Team: {t_player['Squad']}")
    print(f"   Key Metric: {t_player['xG']} xG (Expected Goals)")
    print("=" * 60)

    # 2. Ask AI for the 5 closest matches
    distances, indices = model.kneighbors(target_stats)

    print(f"🤖 AI RECOMMENDATIONS (Sorted by Similarity):")

    for i in range(1, len(indices[0])):
        idx = indices[0][i]
        sim_player = df_clean.iloc[idx]

        # Calculate similarity %
        similarity = max(0, 100 - (distances[0][i] * 15))

        print(f"#{i}: {sim_player['Player']}")
        print(f"    Team: {sim_player['Squad']} ({sim_player['Comp']})")
        print(f"    Similarity Score: {similarity:.1f}%")

        # Display key stats (excluding AerWon as requested)
        print(f"    Stats: {sim_player['xG']} xG | {sim_player['SCA']} Shot Actions | {sim_player['PrgC']} Carries")
        print("-" * 40)

# --- TYPE THE PLAYER NAME BELOW ---
find_bargain_players("Haaland")


🔎 ANALYZING TARGET: Erling Haaland
   Team: Manchester City
   Key Metric: 22.0 xG (Expected Goals)
🤖 AI RECOMMENDATIONS (Sorted by Similarity):
#1: Serhou Guirassy
    Team: Dortmund (de Bundesliga)
    Similarity Score: 78.9%
    Stats: 22.7 xG | 53 Shot Actions | 25 Carries
----------------------------------------
#2: Moise Kean
    Team: Fiorentina (it Serie A)
    Similarity Score: 71.2%
    Stats: 19.4 xG | 57 Shot Actions | 42 Carries
----------------------------------------
#3: Yoane Wissa
    Team: Brentford (eng Premier League)
    Similarity Score: 58.3%
    Stats: 18.5 xG | 69 Shot Actions | 60 Carries
----------------------------------------
#4: Alexander Sørloth
    Team: Atlético Madrid (es La Liga)
    Similarity Score: 55.4%
    Stats: 16.8 xG | 30 Shot Actions | 32 Carries
----------------------------------------
#5: Ante Budimir
    Team: Osasuna (es La Liga)
    Similarity Score: 52.8%
    Stats: 18.3 xG | 46 Shot Actions | 23 Carries
------------------------------